<a href="https://colab.research.google.com/github/shatrill/data_analytics/blob/main/session_6_pandas_data_wrangling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 6: Pandas Data Wrangling

**Course:** Data Analytics Foundation (CompTIA Data+) | **CompTIA Domain:** 2.0 Data Mining | 3.0 Data Analysis

> Open this notebook in **Google Colab**. Data wrangling is like a restaurant kitchen: raw ingredients arrive dirty and in separate crates, and you wash, chop and combine them before anything is served. Tonight you will clean, enrich, combine and summarize data with pandas.

## 🗺️ Your 45-Minute Lab Map
| Part | What you will do | Approx. time |
|---|---|---|
| Part 1 | Learning goals + build a messy purchases table | 4 min |
| Part 2 | Step-by-step cleaning: `strip`, `fillna`, `dropna`, `drop_duplicates` | 10 min |
| Part 3 | Feature engineering with `np.where` + `groupby` summaries | 6 min |
| Part 4 | Guided Example: the SQL → pandas Rosetta Stone and merging tables | 11 min |
| Part 5 | ✏️ Your Turn: the restaurant tips sprint (4 tasks) | 7 min |
| Part 6 | 🧩 Guided Challenge: which shift needs another server? | 10 min |

**How to use this notebook:** read each text cell, then run the code cell beneath it with **Shift + Enter**.
Cells marked **✏️ Your Turn** have blanks (`___`) for you to fill in. Each one has a hidden solution
you can reveal *after* you have tried it. Errors are normal and part of learning. Read the last line
of the error message, fix it, and run the cell again.

## Part 1: What Is a DataFrame, and What Will We Do With It?

A **DataFrame** is pandas' version of a spreadsheet or SQL table: rows are records, and columns are variables. Every column has a data type (numbers, text, dates).

### 🎯 Learning goals
By the end of this lab you can:
1. Clean text, fill or drop missing values, and remove duplicates.
2. Create new calculated columns (feature engineering) with `np.where`.
3. Summarize data with `groupby()` and `.agg()`.
4. Translate SQL ideas (`SELECT`, `WHERE`, `GROUP BY`, `JOIN`) into pandas.
5. Combine tables with `pd.merge()`.

In [1]:
import pandas as pd      # tables
import numpy as np       # fast math and conditional logic

print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 2.2.3 | numpy 2.4.6


### Our messy purchases table
Look for: missing values (`NaN`), stray spaces, inconsistent capitalization, and a duplicated row.

In [4]:
purchases = pd.DataFrame({
    "order_id":  [1, 2, 3, 4, 5, 5, 6, 7, 8, 9],
    "customer":  ["  Alice", "Bob ", "carla", "Dan", "Erin", "Erin", None, "Gus", " hana ", "Ivan"],
    "category":  ["ELECTRONICS", "electronics ", " Home", "home", "Toys", "Toys", "HOME", "Electronics", "toys", "Home "],
    "price":     [499.99, 25.00, 80.00, np.nan, 15.50, 15.50, 60.00, 1200.00, 22.00, np.nan],
    "quantity":  [1, 4, 2, 1, 3, 3, 1, np.nan, 5, 2],
})
purchases

,order_id,customer,category,price,quantity
0,1,Alice,ELECTRONICS,499.99,1.0
1,2,Bob,electronics,25.00,4.0
2,3,carla,Home,80.00,2.0
3,4,Dan,home,NaN,1.0
4,5,Erin,Toys,15.50,3.0
5,5,Erin,Toys,15.50,3.0
6,6,None,HOME,60.00,1.0
7,7,Gus,Electronics,1200.00,NaN
8,8,hana,toys,22.00,5.0
9,9,Ivan,Home,NaN,2.0


## Part 2: Step-by-Step Cleaning

### Step 1: Strip spaces and standardize capitalization
`.str.strip()` removes spaces at the edges; `.str.title()` makes "electronics " and "ELECTRONICS" both become "Electronics".

In [5]:
print("Before:", purchases["category"].unique())

purchases["customer"] = purchases["customer"].str.strip().str.title()
purchases["category"] = purchases["category"].str.strip().str.title()

print("After: ", purchases["category"].unique())

Before: ['ELECTRONICS' 'electronics ' ' Home' 'home' 'Toys' 'HOME' 'Electronics'
 'toys' 'Home ']
After:  ['Electronics' 'Home' 'Toys']


### Step 2: Remove duplicate rows
Order 5 appears twice, which would double-count Erin's revenue.

In [6]:
print("Duplicates:", purchases.duplicated().sum())
purchases = purchases.drop_duplicates()
print("Rows now:", len(purchases))

Duplicates: 1
Rows now: 9


### Step 3: Handle missing values: impute vs. drop
Check what is missing first, then make a **business decision for each column**:

| Column | Decision | Why |
|---|---|---|
| `price` | **Impute** with the median price of the same category | We can make a reasonable estimate |
| `quantity` | **Impute** with 1 | Business rule: a missing quantity means a single item |
| `customer` | **Drop** the row | We cannot guess who bought it, and our report is by customer |

In [ ]:
purchases.isnull().sum()

In [8]:
# Price: fill with the median price of the SAME category (smarter than one global median)
purchases["price"] = purchases["price"].fillna(
    purchases.groupby("category")["price"].transform("median"))

# Quantity: business rule, a missing quantity means 1
purchases["quantity"] = purchases["quantity"].fillna(1).astype(int)

# Customer: a critical field, so drop rows where it is missing
purchases = purchases.dropna(subset=["customer"])

print(purchases.isnull().sum().sum(), "missing values remain")
purchases

0 missing values remain


,order_id,customer,category,price,quantity
0,1,Alice,Electronics,499.99,1
1,2,Bob,Electronics,25.00,4
2,3,Carla,Home,80.00,2
3,4,Dan,Home,70.00,1
4,5,Erin,Toys,15.50,3
7,7,Gus,Electronics,1200.00,1
8,8,Hana,Toys,22.00,5
9,9,Ivan,Home,70.00,2


💡 `.transform("median")` calculates the median **per category** and lines it up with every row, so each missing price gets its own category's typical price.

## Part 3: Feature Engineering and Group Summaries

**Feature engineering** = creating new columns that reveal insights.

### Math columns and `np.where` (IF / THEN / ELSE in one line)
`np.where(condition, value_if_true, value_if_false)` works just like Excel's `IF()`.

In [9]:
purchases["revenue"] = purchases["price"] * purchases["quantity"]

purchases["customer_tier"] = np.where(purchases["revenue"] >= 200, "High Value", "Standard")
purchases[["customer", "category", "revenue", "customer_tier"]]

,customer,category,revenue,customer_tier
0,Alice,Electronics,499.99,High Value
1,Bob,Electronics,100.00,Standard
2,Carla,Home,160.00,Standard
3,Dan,Home,70.00,Standard
4,Erin,Toys,46.50,Standard
7,Gus,Electronics,1200.00,High Value
8,Hana,Toys,110.00,Standard
9,Ivan,Home,140.00,Standard


### `groupby()` summaries
`groupby` = **split** the rows into groups → **apply** a calculation → **combine** the results. Named aggregation (`new_name=("column", "function")`) gives tidy, labeled output.

In [10]:
category_summary = (purchases
                    .groupby("category")
                    .agg(orders=("order_id", "count"),
                         units=("quantity", "sum"),
                         revenue=("revenue", "sum"),
                         avg_price=("price", "mean"))
                    .round(2)
                    .sort_values("revenue", ascending=False))
category_summary

,orders,units,revenue,avg_price
category,,,,
Electronics,3,6,1799.99,575.00
Home,3,5,370.00,73.33
Toys,2,8,156.50,18.75


## Part 4: Guided Example: The SQL → Pandas Rosetta Stone and Merging Tables

You already speak SQL from Sessions 3–4. Pandas does the same jobs with different words:

| Job | SQL | pandas |
|---|---|---|
| Pick columns | `SELECT a, b` | `df[["a", "b"]]` |
| Filter rows | `WHERE a > 10` | `df[df["a"] > 10]` |
| Sort | `ORDER BY a DESC` | `df.sort_values("a", ascending=False)` |
| Summarize | `GROUP BY a` | `df.groupby("a").agg(...)` |
| Join | `LEFT JOIN t2 ON ...` | `pd.merge(df, t2, on="key", how="left")` |

### Step 1: A second table, customer profiles
Real data lives in separate tables. Note that **Ivan has no profile** and **Zoe has a profile but no purchases**.

In [11]:
profiles = pd.DataFrame({
    "customer":     ["Alice", "Bob", "Carla", "Dan", "Erin", "Gus", "Hana", "Zoe"],
    "city":         ["Rochester", "Buffalo", "Rochester", "Syracuse", "Albany", "Rochester", "Buffalo", "Ithaca"],
    "member_since": [2019, 2022, 2021, 2023, 2020, 2018, 2024, 2022],
})
profiles

,customer,city,member_since
0,Alice,Rochester,2019
1,Bob,Buffalo,2022
2,Carla,Rochester,2021
3,Dan,Syracuse,2023
4,Erin,Albany,2020
5,Gus,Rochester,2018
6,Hana,Buffalo,2024
7,Zoe,Ithaca,2022


### Step 2: SELECT + WHERE + ORDER BY
> SQL: `SELECT customer, category, revenue FROM purchases WHERE revenue > 50 ORDER BY revenue DESC`

In [12]:
(purchases[purchases["revenue"] > 50]
          [["customer", "category", "revenue"]]
          .sort_values("revenue", ascending=False))

,customer,category,revenue
7,Gus,Electronics,1200.00
0,Alice,Electronics,499.99
2,Carla,Home,160.00
9,Ivan,Home,140.00
8,Hana,Toys,110.00
1,Bob,Electronics,100.00
3,Dan,Home,70.00


### Step 3: INNER vs. LEFT merge, and always check your row counts
- **inner**: only customers found in *both* tables.
- **left**: *all* purchases are kept, with profile details added where a match exists.

In [16]:
inner = pd.merge(purchases, profiles, on="customer", how="inner")
left  = pd.merge(purchases, profiles, on="customer", how="left")

print("purchases rows:", len(purchases))
print("inner merge rows:", len(inner), " <- Ivan's purchase was dropped!")
print("left  merge rows:", len(left))
left[["customer", "category", "revenue", "city", "member_since"]]

purchases rows: 8
inner merge rows: 7  <- Ivan's purchase was dropped!
left  merge rows: 8


,customer,category,revenue,city,member_since
0,Alice,Electronics,499.99,Rochester,2019.0
1,Bob,Electronics,100.00,Buffalo,2022.0
2,Carla,Home,160.00,Rochester,2021.0
3,Dan,Home,70.00,Syracuse,2023.0
4,Erin,Toys,46.50,Albany,2020.0
5,Gus,Electronics,1200.00,Rochester,2018.0
6,Hana,Toys,110.00,Buffalo,2024.0
7,Ivan,Home,140.00,NaN,NaN


### Step 4: Find the unmatched rows with `indicator=True`
This adds a `_merge` column that says where each row came from. It is a great data-quality check.

In [14]:
check = pd.merge(purchases, profiles, on="customer", how="outer", indicator=True)
check["_merge"].value_counts()

,count
_merge,
both,7
left_only,1
right_only,1


In [17]:
check[check["_merge"] != "both"][["customer", "order_id", "city", "_merge"]]

,customer,order_id,city,_merge
7,Ivan,9.0,NaN,left_only
8,Zoe,NaN,Ithaca,right_only


**Read it:** `left_only` = Ivan bought something but has no profile. `right_only` = Zoe has a profile but never purchased, which makes her a marketing re-engagement target!

### Step 5: Merge, fill gaps, then GROUP BY
> SQL: `SELECT COALESCE(city,'Unknown') AS city, SUM(revenue) ... LEFT JOIN ... GROUP BY city`

In [18]:
left["city"] = left["city"].fillna("Unknown")     # pandas' version of COALESCE

(left.groupby("city")
     .agg(customers=("customer", "nunique"), revenue=("revenue", "sum"))
     .sort_values("revenue", ascending=False))

,customers,revenue
city,,
Rochester,3,1859.99
Buffalo,2,210.00
Unknown,1,140.00
Syracuse,1,70.00
Albany,1,46.50


## Part 5: ✏️ Your Turn: The Restaurant Tips Sprint

Load the public **tips** dataset (244 restaurant bills) and complete the four sprint tasks from the slides.

In [21]:
# TASK 1: load the data and inspect its structure
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
tips = pd.read_csv(url)
tips.info()        # shows columns, data types and missing values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   total_bill  244 non-null    float64
 1   tip         244 non-null    float64
 2   sex         244 non-null    object 
 3   smoker      244 non-null    object 
 4   day         244 non-null    object 
 5   time        244 non-null    object 
 6   size        244 non-null    int64  
dtypes: float64(2), int64(1), object(4)
memory usage: 13.5+ KB


<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
tips = pd.read_csv(url)
tips.info()
```

</details>

In [22]:
# TASK 2: standardize the 'sex' and 'smoker' columns to UPPERCASE
tips["sex"]    = tips["sex"].str.upper()
tips["smoker"] = tips["smoker"].str.upper()
tips.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,FEMALE,NO,Sun,Dinner,2
1,10.34,1.66,MALE,NO,Sun,Dinner,3
2,21.01,3.50,MALE,NO,Sun,Dinner,3
3,23.68,3.31,MALE,NO,Sun,Dinner,2
4,24.59,3.61,FEMALE,NO,Sun,Dinner,4


<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
tips["sex"]    = tips["sex"].str.upper()
tips["smoker"] = tips["smoker"].str.upper()
tips.head()
```

</details>

In [23]:
# TASK 3: create tip_percentage = (tip / total_bill) * 100, rounded to 1 decimal
tips["tip_percentage"] = (tips["tip"] / tips["total_bill"] * 100).round(1)
tips[["total_bill", "tip", "tip_percentage"]].head()

,total_bill,tip,tip_percentage
0,16.99,1.01,5.9
1,10.34,1.66,16.1
2,21.01,3.50,16.7
3,23.68,3.31,14.0
4,24.59,3.61,14.7


<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
tips["tip_percentage"] = (tips["tip"] / tips["total_bill"] * 100).round(1)
tips[["total_bill", "tip", "tip_percentage"]].head()
```

</details>

In [24]:
# TASK 4: group by day AND time. Which shift earns the highest average tip?
shift_tips = (tips.groupby(["day", "time"])
                  .agg(bills=("tip", "count"), avg_tip=("tip", "mean"), avg_tip_pct=("tip_percentage", "mean"))
                  .round(2)
                  .sort_values("avg_tip", ascending=False))
shift_tips

,,bills,avg_tip,avg_tip_pct
day,time,,,
Sun,Dinner,76,3.26,16.69
Thur,Dinner,1,3.00,16.00
Sat,Dinner,87,2.99,15.31
Fri,Dinner,12,2.94,15.90
Thur,Lunch,61,2.77,16.13
Fri,Lunch,7,2.38,18.89


<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
shift_tips = (tips.groupby(["day", "time"])
                  .agg(bills=("tip", "count"), avg_tip=("tip", "mean"), avg_tip_pct=("tip_percentage", "mean"))
                  .round(2)
                  .sort_values("avg_tip", ascending=False))
shift_tips
```

</details>

## Part 6: 🧩 Guided Challenge: Which Shift Needs Another Server?

The restaurant manager gives you this week's **staffing schedule**. Combine it with the tips data to find which shift's servers earn the most tips *per person*, and where the workload per server is heaviest.

In [25]:
staffing = pd.DataFrame({
    "day":     ["Thur", "Thur", "Fri", "Fri", "Sat", "Sun"],
    "time":    ["Lunch", "Dinner", "Lunch", "Dinner", "Dinner", "Dinner"],
    "servers": [3, 1, 2, 3, 4, 4],
})
staffing

,day,time,servers
0,Thur,Lunch,3
1,Thur,Dinner,1
2,Fri,Lunch,2
3,Fri,Dinner,3
4,Sat,Dinner,4
5,Sun,Dinner,4


**Step 1: Bill-size categories with `np.select`** (a multi-level `np.where`). Label bills as `"Small"` (< $15), `"Medium"` ($15–$30), or `"Large"` (> $30). Conditions are checked in order.

In [26]:
conditions = [tips["total_bill"] < 15,
              tips["total_bill"] <= 30]
labels     = ["Small", "Medium"]
tips["bill_size"] = np.select(conditions, labels, default="Large")

tips["bill_size"].value_counts()

,count
bill_size,
Medium,132
Small,80
Large,32


<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
conditions = [tips["total_bill"] < 15,
              tips["total_bill"] <= 30]
labels     = ["Small", "Medium"]
tips["bill_size"] = np.select(conditions, labels, default="Large")
tips["bill_size"].value_counts()
```

</details>

**Step 2: Summarize per shift.** For each `day` + `time`, compute the number of bills, the total tips, and how many bills were `"Large"`.
Use `.reset_index()` at the end so `day` and `time` become normal columns, ready to merge.

In [27]:
shift = (tips.groupby(["day", "time"])
             .agg(bills=("tip", "count"),
                  total_tips=("tip", "sum"),
                  large_bills=("bill_size", lambda s: (s == "Large").sum()))
             .reset_index())
shift

,day,time,bills,total_tips,large_bills
0,Fri,Dinner,12,35.28,1
1,Fri,Lunch,7,16.68,0
2,Sat,Dinner,87,260.40,12
3,Sun,Dinner,76,247.39,14
4,Thur,Dinner,1,3.00,0
5,Thur,Lunch,61,168.83,5


<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
shift = (tips.groupby(["day", "time"])
             .agg(bills=("tip", "count"),
                  total_tips=("tip", "sum"),
                  large_bills=("bill_size", lambda s: (s == "Large").sum()))
             .reset_index())
shift
```

</details>

**Step 3: Merge in the staffing schedule** on **two** keys (`day` and `time`), then calculate `tips_per_server` and `bills_per_server`.

In [28]:
shift = pd.merge(shift, staffing, on=["day", "time"], how="left")

shift["tips_per_server"]  = (shift["total_tips"] / shift["servers"]).round(2)
shift["bills_per_server"] = (shift["bills"] / shift["servers"]).round(1)

shift.sort_values("bills_per_server", ascending=False)

,day,time,bills,total_tips,large_bills,servers,tips_per_server,bills_per_server
2,Sat,Dinner,87,260.40,12,4,65.10,21.8
5,Thur,Lunch,61,168.83,5,3,56.28,20.3
3,Sun,Dinner,76,247.39,14,4,61.85,19.0
0,Fri,Dinner,12,35.28,1,3,11.76,4.0
1,Fri,Lunch,7,16.68,0,2,8.34,3.5
4,Thur,Dinner,1,3.00,0,1,3.00,1.0


<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
shift = pd.merge(shift, staffing, on=["day", "time"], how="left")
shift["tips_per_server"]  = (shift["total_tips"] / shift["servers"]).round(2)
shift["bills_per_server"] = (shift["bills"] / shift["servers"]).round(1)
shift.sort_values("bills_per_server", ascending=False)
```

</details>

**Step 4: A pivot table for the manager.** `pivot_table` reshapes the data into a grid: days as rows, times as columns, and average tip % in the cells.

In [30]:
tips.pivot_table(index="day", columns="time", values="tip_percentage", aggfunc="mean").round(1)

time,Dinner,Lunch
day,,
Fri,15.9,18.9
Sat,15.3,NaN
Sun,16.7,NaN
Thur,16.0,16.1


<details>
<summary><b>💡 Reveal a possible solution</b> (try it yourself first!)</summary>

```python
tips.pivot_table(index="day", columns="time", values="tip_percentage", aggfunc="mean").round(1)
```

</details>

**✍️ Your recommendation (double-click to edit):** Which shift should get an extra server, and why? Use the `bills_per_server` and `large_bills` numbers as evidence.

<details>
<summary><b>💡 Reveal a sample answer</b></summary>

**Saturday dinner** is the most stretched shift (about 22 bills per server, plus 12 large bills), with **Thursday lunch** (about 20) and **Sunday dinner** (19, and the most large bills at 14) close behind. Friday dinner has 3 servers for only 12 bills, so moving one Friday server to Saturday dinner improves coverage without adding payroll. *Caveat: the tips data is a sample of bills, so confirm with full sales records before changing the schedule.*

</details>

---
## ✅ Lab Checklist
- [ ] Cleaned text, imputed/dropped missing values, removed duplicates
- [ ] Created calculated columns with `np.where` and `np.select`
- [ ] Summarized with `groupby().agg()` and `pivot_table`
- [ ] Translated SQL to pandas and compared inner vs. left merges
- [ ] Completed the 4-task tips sprint
- [ ] Merged staffing data and made a data-backed staffing recommendation

## 💾 Save Your Work to GitHub
1. In Colab, click **File → Save a copy in GitHub**.
2. Choose your course/Capstone repository.
3. Use this commit message: `Session 6: Completed Pandas Wrangling & Feature Engineering Lab`

**Coming up next:** Session 7: Exploratory Data Analysis: distributions, outliers, and your first charts with Matplotlib and Seaborn.